# 04 — Cross-Dataset Generalisation (CICIoT2023)
## Response to IEEE Access Reviewer 1, Comment 6

Fatma Mohammed Dhaou · University of Tabuk

**Purpose.** Reviewer 1 asks whether the conclusions generalise beyond Edge-IIoTset. This notebook applies the *same diagnostic protocol* to CICIoT2023 — the direct 2023 successor benchmark — to test whether the shortcut phenomenon is a general property of testbed-derived IoT/IIoT benchmarks rather than an Edge-IIoTset artefact.

**This is a focused diagnostic, not a full benchmark study.** The protocol is exactly four steps:
1. Linear probe on the binary task — does a hyperplane achieve near-perfect separation?
2. If so, correlation + mutual-information screen — which features carry it?
3. Remove the flagged features.
4. Non-linear survival — does Random Forest retain near-perfect accuracy?

If steps 1 and 4 both hold on a second, independently constructed dataset, the methodological claim generalises.

---
### Getting the data
CICIoT2023 is large (169 CSV files). You do **not** need all of it. Download from the University of New Brunswick (https://www.unb.ca/cic/datasets/iotdataset-2023.html) or Kaggle, and place a handful of the per-attack CSV parts in a folder. This notebook loads a **stratified sample** capped at ~300k rows, which is ample for the diagnostic and fits Colab memory.

*Fallback:* if CICIoT2023 is intractable on your connection, N-BaIoT (much smaller) works with the same protocol — set `FALLBACK_NBAIOT = True` and point `DATA_GLOB` at its CSVs.


In [ ]:
import glob, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
RS=42; np.random.seed(RS)

# Colab: upload the CSV part(s) or mount Drive, then set the glob pattern:
DATA_GLOB = 'CICIoT2023/*.csv'      # <-- adjust to where your CSV parts are
SAMPLE_CAP = 300_000

paths = sorted(glob.glob(DATA_GLOB))
print(f'Found {len(paths)} CSV files')
assert paths, 'No files matched DATA_GLOB — adjust the path.'

frames=[]
for p in paths:
    frames.append(pd.read_csv(p))
raw = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(raw):,} rows x {raw.shape[1]} cols')
# stratified down-sample if huge
label_col = [c for c in raw.columns if c.lower() in ('label','attack','class')][0]
if len(raw) > SAMPLE_CAP:
    raw = raw.groupby(label_col, group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(50, int(SAMPLE_CAP*len(g)/len(raw)))), random_state=RS))
print(f'Working sample: {len(raw):,} rows; label column = {label_col!r}')


### Step 0 — Binary label and clean feature matrix


In [ ]:
BENIGN_TOKENS = ['benign','benigntraffic','normal']
lab = raw[label_col].astype(str).str.lower()
yb = (~lab.isin(BENIGN_TOKENS)).astype(int)   # 1 = attack, 0 = benign
print('Attack rate:', yb.mean().round(3), '| benign rows:', (yb==0).sum())

Xc = raw.drop(columns=[label_col]).select_dtypes(include=[np.number]).copy()
Xc = Xc.replace([np.inf,-np.inf], np.nan).dropna(axis=1, how='any')
Xc = Xc.loc[:, Xc.nunique() > 1]
# align rows after any column drops
Xc = Xc.reset_index(drop=True); yb = yb.reset_index(drop=True)
print(f'Numeric feature matrix: {Xc.shape[1]} features')


### Step 1 — Linear probe (does a hyperplane separate near-perfectly?)


In [ ]:
Xtr,Xte,ytr,yte = train_test_split(Xc, yb, test_size=.2, random_state=RS, stratify=yb)
sc = StandardScaler().fit(Xtr)
lin = LogisticRegression(penalty='l2', max_iter=2000, random_state=RS).fit(sc.transform(Xtr), ytr)
lin_acc = accuracy_score(yte, lin.predict(sc.transform(Xte)))
lin_auc = roc_auc_score(yte, lin.predict_proba(sc.transform(Xte))[:,1])
print(f'Linear probe: accuracy={lin_acc:.4f}  AUC={lin_auc:.4f}')
print('Near-1.0 => a hyperplane separates attacks from benign — a shortcut signature.')


### Step 2 — Screen (correlation + mutual information, train only)


In [ ]:
corr = Xtr.corrwith(ytr).abs().sort_values(ascending=False)
mi = pd.Series(mutual_info_classif(Xtr, ytr, random_state=RS), index=Xtr.columns).sort_values(ascending=False)
print('Top 8 |Pearson r|:'); print(corr.head(8).to_string(float_format=lambda v:f'{v:.4f}'))
print('\nTop 8 Mutual Information:'); print(mi.head(8).to_string(float_format=lambda v:f'{v:.4f}'))

FLAG = sorted(set(corr[corr>0.85].index) | set(mi.head(4).index))
print(f'\nFlagged (|r|>0.85 OR MI top-4): {FLAG}')


### Steps 3–4 — Remove flagged features, test non-linear survival


In [ ]:
keep = [c for c in Xc.columns if c not in FLAG]
rf_ctrl = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1).fit(Xtr[keep], ytr)
rf_acc = accuracy_score(yte, rf_ctrl.predict(Xte[keep]))
rf_f1  = f1_score(yte, rf_ctrl.predict(Xte[keep]))
print(f'After removing {len(FLAG)} flagged features ({len(keep)} left):')
print(f'Random Forest accuracy={rf_acc:.4f}  F1={rf_f1:.4f}')
print()
print('INTERPRETATION')
print('-'*60)
print(f'Linear probe (all features):     {lin_acc:.4f}')
print(f'Random Forest (flagged removed): {rf_acc:.4f}')
print('If the linear probe was near-perfect AND RF stays near-perfect after')
print('the flagged features are removed, CICIoT2023 exhibits the SAME redundant')
print('shortcut pattern as Edge-IIoTset — the methodological claim generalises.')


---

Copy this output back. Two numbers matter for the manuscript: the **linear-probe accuracy** (step 1) and the **Random-Forest-after-removal accuracy** (step 4). Together they establish whether the phenomenon replicates.
